# **Import Dependencies**

In [23]:
import os
import gymnasium as gym
from stable_baselines3 import PPO 
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import (
    CheckpointCallback,
    EvalCallback,
    StopTrainingOnRewardThreshold,
    CallbackList
)
from pathlib import Path

# **Load Environment**

Applying a Penalty for using Side Thrusters, when both legs in contact for a certain amount of time

In [115]:
environment_name = "LunarLander-v2"

test_env = gym.make(environment_name, 
               render_mode="human",
               continuous=False, 
               gravity=-10.0,
               enable_wind=False, 
               wind_power=15.0, 
               turbulence_power=1.5)

train_env = SubprocVecEnv([
    lambda: gym.make(environment_name, 
               render_mode="None",
               continuous=False, 
               gravity=-10.0,
               enable_wind=False, 
               wind_power=15.0, 
               turbulence_power=1.5)
    for _ in range(4)
])

In [25]:
episodes = 2
for episode in range(1, episodes + 1):  # Looping through the environment 5 times
    state, info = test_env.reset()
    terminated = False  # Tracks natural episode completion
    truncated = False   # Tracks forced episode termination (e.g., time limit)
    score = 0

    while not (terminated or truncated):  # Stop when either condition is met
        test_env.render()
        action = test_env.action_space.sample()
        n_state, reward, terminated, truncated, info = test_env.step(action)  # Updated unpacking
        score += reward

    print(f"Episode: {episode} Score: {score}")

Episode: 1 Score: -69.25435447841376
Episode: 2 Score: -227.71256939316726


# **Train an RL**

In [116]:
save_path = Path(r"C:\Users\KIIT\OneDrive\Desktop\VS CODE\Reinforcement\Lunar_Lander\saved_models")
checkpoint_path = Path(r"C:\Users\KIIT\OneDrive\Desktop\VS CODE\Reinforcement\Lunar_Lander\checkpoints")

In [117]:
stop_callback = StopTrainingOnRewardThreshold(
    reward_threshold=250,
    verbose=1
)

eval_callback = EvalCallback(
    test_env,
    callback_on_new_best=stop_callback,
    eval_freq=5000,
    best_model_save_path=save_path,
    verbose=1
)

checkpoint_callback = CheckpointCallback(
    save_freq=2000,          # every 10k steps
    save_path=checkpoint_path,
    name_prefix="ppo_lunar",
    verbose=2
)

callback = CallbackList([checkpoint_callback, eval_callback])

In [118]:
model = PPO(
    "MlpPolicy", 
    train_env,
    verbose=1,

    learning_rate=3e-4,
    gamma=0.99, 
    gae_lambda=0.95,

    ent_coef=0.01, 
    vf_coef=0.5,
    clip_range=0.2, 
    max_grad_norm=0.5
)

Using cpu device


In [119]:
model.learn(total_timesteps=20000, callback=callback)

c:\Users\KIIT\OneDrive\Desktop\VS CODE\Reinforcement\Lunar_Lander\RL_Lunar_Lander\.venv\lib\site-packages\stable_baselines3\common\callbacks.py:414: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.subproc_vec_env.SubprocVecEnv object at 0x000001677BC41960> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x000001677BC41E40>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


Saving model checkpoint to C:\Users\KIIT\OneDrive\Desktop\VS CODE\Reinforcement\Lunar_Lander\checkpoints\ppo_lunar_8000_steps.zip
-----------------------------
| time/              |      |
|    fps             | 2206 |
|    iterations      | 1    |
|    time_elapsed    | 3    |
|    total_timesteps | 8192 |
-----------------------------
Saving model checkpoint to C:\Users\KIIT\OneDrive\Desktop\VS CODE\Reinforcement\Lunar_Lander\checkpoints\ppo_lunar_16000_steps.zip
-----------------------------------------
| time/                   |             |
|    fps                  | 1228        |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.008580141 |
|    clip_fraction        | 0.0571      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | 0.0031      |
|    learning_rate       

In [ ]:
model.save(save_path / "saved_model.zip")

# **Evaluation and Testing**

In [53]:
import os
path = r"C:\Users\KIIT\OneDrive\Desktop\VS CODE\Reinforcement\Lunar_Lander\checkpoints"  # example
if os.path.exists(path):
    print("✅ File exists")
else:
    print(" File not found")

✅ File exists


In [ ]:
model = PPO.load(r"C:\Users\KIIT\OneDrive\Desktop\VS CODE\Reinforcement\Lunar_Lander\saved_models.zip")
model.set_env(test_env)

In [35]:
evaluate_policy(model , test_env , n_eval_episodes=3 , render=False )

(-742.551748965343, 145.25820548309468)

In [ ]:
episodes = 5

for episode in range(1, episodes + 1):
    obs, info = test_env.reset()
    terminated = False
    truncated = False
    score = 0

    while not (terminated or truncated):
        test_env.render()
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = test_env.step(action)
        score += reward

    print(f"Episode {episode} - Score: {score}")

test_env.close()

In [120]:
test_env.close() 
train_env.close()